# PromptMind — Semantic Retrieval Pipeline
**Stage 1:** Parse PDFs → Embed → Build FAISS index → Save to disk  
**Stage 2:** Load saved index → Query → Groq LLM answer

In [1]:
import os
import json
import pickle
from datetime import datetime
from typing import List, Dict
import fitz
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

False

## Load Sentence Transformer Model

In [2]:
MODEL_PATH = "sentence_transformer_models/all-MiniLM-L6-v2"

# Download and save locally if not already saved
if not os.path.exists(MODEL_PATH):
    print("Downloading model from HuggingFace...")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    model.save(MODEL_PATH)
    print("Model saved locally.")
else:
    model = SentenceTransformer(MODEL_PATH, device="cpu")
    print("Model loaded from local path.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded from local path.


## Stage 1 — Pipeline Functions

In [3]:
def extract_chunks_from_pdf(file_path: str) -> List[Dict]:
    doc = fitz.open(file_path)
    chunks = []
    for i, page in enumerate(doc):
        blocks = page.get_text("blocks")
        for b in blocks:
            text = b[4].strip()
            if len(text) > 100:
                chunks.append({
                    "text": text,
                    "page_number": i + 1,
                    "document": os.path.basename(file_path)
                })
    doc.close()
    return chunks

In [4]:
def build_faiss_index(chunks: List[Dict]):
    print(f"Embedding {len(chunks)} chunks...")
    embeddings = model.encode([c['text'] for c in chunks], show_progress_bar=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.array(embeddings, dtype='float32'))
    # Store embedding in chunk for reference
    for i, emb in enumerate(embeddings):
        chunks[i]['embedding'] = emb
    return index, chunks

In [5]:
def semantic_search(query: str, index, chunks, top_k=5):
    query_embedding = model.encode([query])
    D, I = index.search(np.array(query_embedding, dtype='float32'), top_k)
    return [chunks[idx] for idx in I[0]]

In [6]:
def build_output_json(input_docs, persona, task, top_chunks):
    now = datetime.now().isoformat()
    return {
        "metadata": {
            "input_documents": input_docs,
            "persona": persona,
            "job_to_be_done": task,
            "processing_timestamp": now
        },
        "extracted_sections": [
            {
                "document": c['document'],
                "page_number": c['page_number'],
                "section_title": c['text'][:100],
                "importance_rank": i + 1
            }
            for i, c in enumerate(top_chunks)
        ],
        "subsection_analysis": [
            {
                "document": c['document'],
                "refined_text": c['text'],
                "page_number": c['page_number']
            }
            for c in top_chunks
        ]
    }

In [7]:
def run_pipeline(folder_paths: List[str], persona: str, task: str, top_k: int = 5) -> Dict:
    all_chunks = []
    input_files = []

    for folder in folder_paths:
        for fname in os.listdir(folder):
            if fname.endswith(".pdf"):
                path = os.path.join(folder, fname)
                chunks = extract_chunks_from_pdf(path)
                all_chunks.extend(chunks)
                input_files.append(fname)

    faiss_index, enriched_chunks = build_faiss_index(all_chunks)

    query = f"{persona} wants to: {task}"
    top_chunks = semantic_search(query, faiss_index, enriched_chunks, top_k)

    result_json = build_output_json(input_files, persona, task, top_chunks)

    # ── NEW: return index + chunks so we can save them ──
    return result_json, faiss_index, enriched_chunks

## Run Pipeline & Save Index

In [8]:
# ==== Configure your folders and query here ====
folders = ["Cook/", "Travel/", "HR/"]
persona = "HR manager"
task    = "What are the benefits of generative AI?"

output, faiss_index, enriched_chunks = run_pipeline(folders, persona, task)

# Save pipeline output JSON
with open("pipeline_output.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

# ── NEW: Save FAISS index and chunks to disk ──
faiss.write_index(faiss_index, "index.faiss")

# Remove embeddings before pickling (they're numpy arrays, not needed for text retrieval)
chunks_to_save = [{k: v for k, v in c.items() if k != 'embedding'} for c in enriched_chunks]
with open("chunks.pkl", "wb") as f:
    pickle.dump(chunks_to_save, f)

print("✅ Pipeline done. Saved: pipeline_output.json, index.faiss, chunks.pkl")

Embedding 1737 chunks...


Batches:   0%|          | 0/55 [00:00<?, ?it/s]

✅ Pipeline done. Saved: pipeline_output.json, index.faiss, chunks.pkl


## Stage 2 — Ask Questions (No Re-indexing Needed)
From here down, the pipeline never runs again. Just load the saved index and query.

In [9]:
# Load saved index and chunks
index = faiss.read_index("index.faiss")
with open("chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

print(f"✅ Loaded index with {index.ntotal} vectors and {len(chunks)} chunks")

✅ Loaded index with 1737 vectors and 1737 chunks


In [ ]:
# Groq LLM setup
groq_client = Groq(api_key="your-api-key")

def generate_answer(query: str, top_k: int = 5) -> str:
    # Step 1: Retrieve relevant chunks
    query_vec = model.encode([query]).astype("float32")
    _, I = index.search(query_vec, top_k)
    top_chunks = [chunks[i] for i in I[0]]

    # Step 2: Build context
    context = "\n\n".join([
        f"[{c['document']} — Page {c['page_number']}]\n{c['text'][:600]}"
        for c in top_chunks
    ])

    # Step 3: Ask Groq
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "Answer using only the provided context. Be concise. Always cite the document name and page number."
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {query}"
            }
        ],
        max_tokens=1024
    )

    answer = response.choices[0].message.content.strip()

    # Also print sources
    print("\n📄 Sources used:")
    for c in top_chunks:
        print(f"  - {c['document']} — Page {c['page_number']}")

    return answer

In [16]:
# ==== Ask anything ====
query = "What are the benefits of generative AI?"
answer = generate_answer(query)
print("\n🧠 Answer:")
print(answer)


📄 Sources used:
  - Learn Acrobat - Generative AI_1.pdf — Page 19
  - Learn Acrobat - Generative AI_1.pdf — Page 20
  - Learn Acrobat - Generative AI_1.pdf — Page 19
  - Learn Acrobat - Generative AI_1.pdf — Page 6
  - Learn Acrobat - Generative AI_1.pdf — Page 3

🧠 Answer:
The document does not explicitly state the benefits of generative AI, but it implies that the features can produce useful content (Learn Acrobat - Generative AI_1.pdf — Page 19) and provide responses to specific queries (Learn Acrobat - Generative AI_1.pdf — Page 20).
